## Import

In [4]:
import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

# Add src/ to path (once, so imports work)
sys.path.append(str(Path().resolve().parent / "src"))
# 
# Enable autoreload for Jupyter notebooks
%load_ext autoreload
%autoreload 2

from paths import DATA_DATASETS

Failed to read module file 'C:\Users\JuliusAdmin\AppData\Local\Programs\Python\Python312\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\JuliusAdmin\Documents\GitHub\Marketing-Analytics\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\JuliusAdmin\Documents\GitHub\Marketing-Analytics\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\JuliusAdmin\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in

In [14]:
# Load data
transactions_processed = pd.read_csv(DATA_DATASETS / "transactions_processed.csv", sep=";", parse_dates=["date"])
transactions = pd.read_csv(DATA_DATASETS / "transactions.csv", sep=";")
baseline = pd.read_csv(DATA_DATASETS / "pred.csv")
baseline = baseline.rename(columns={"Id": "customer.id", "predicted.period.CLV": "clv_baseline"})

In [15]:
print(transactions["rentalPeriod.start"].min())
print(transactions["rentalPeriod.start"].max())

2016-03-11
2023-09-05


In [16]:
print(transactions_processed["date"].min())
print(transactions_processed["date"].max())

2016-03-11 00:00:00
2023-09-05 00:00:00


In [ ]:
# Filter transactions to the 12 months following the training period and calculate actual CLV
actuals = (
    transactions_processed[
        (transactions_processed["date"] >= "2023-09-06") &
        (transactions_processed["date"] <= "2024-09-06")
    ]
    .groupby("customer.id")["revenue"]
    .sum()
    .reset_index()
    .rename(columns={"revenue": "actual_clv"})
)

In [7]:
# Merge baseline predictions with actual CLV
df = baseline[["customer.id", "clv_baseline"]].merge(actuals, on="customer.id", how="left")
df["actual_clv"] = df["actual_clv"].fillna(0)

In [ ]:
# Metrics
def evaluate(y_true, y_pred, name):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    spearman = stats.spearmanr(y_true, y_pred).correlation
    print(f"{name}:")
    print(f"  MAE      = {mae:.2f}")
    print(f"  RMSE     = {rmse:.2f}")
    print(f"  Spearman = {spearman:.3f}")

evaluate(df["actual_clv"], df["clv_baseline"], "CLVTools Baseline")

CLVTools Baseline:
  MAE      = 2338.65
  RMSE     = 9545.50
  Spearman = nan


C:\Users\JuliusAdmin\AppData\Local\Temp\ipykernel_30488\601713657.py:5: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman = stats.spearmanr(y_true, y_pred).correlation


In [9]:
print(f"Total customers: {len(df)}")
print(f"Customers with actual > 0: {(df['actual_clv'] > 0).sum()}")
print(f"Ø actual CLV: {df['actual_clv'].mean():.2f}")
print(f"Ø predicted CLV: {df['clv_baseline'].mean():.2f}")

Total customers: 7245
Customers with actual > 0: 0
Ø actual CLV: 0.00
Ø predicted CLV: 2338.65
